In [6]:
import pandas as pd
from sqlalchemy import create_engine, types, text
# import psycopg
import os

In [7]:
db_url = os.environ['DB_URL']
engine = create_engine(db_url, echo=True)

In [8]:
prefix='data'
data={
    # 'preprocessed_data': f'{prefix}/preprocessed_test.csv',
    # 'application_train': f'{prefix}/application_train.csv',
    'application_test': f'{prefix}/application_test.csv',
    'bureau': f'{prefix}/bureau.csv',
    'bureau_balance': f'{prefix}/bureau_balance.csv',
    'previous_application': f'{prefix}/previous_application.csv',
    'POS_CASH_balance': f'{prefix}/POS_CASH_balance.csv',
    'installments_payments': f'{prefix}/installments_payments.csv',
    'credit_card_balance': f'{prefix}/credit_card_balance.csv'
}

In [9]:
bureau_df = pd.read_csv(data['bureau'])
allowed_bureau_ids = bureau_df[bureau_df['SK_ID_CURR'] <= 105000]['SK_ID_BUREAU'].unique()

for name, file in data.items():
    print(f"===============\nProcessing table: {name}\n===============")
    df = pd.read_csv(file)
    
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    if 'SK_ID_CURR' in df.columns:
        df = df[df['SK_ID_CURR']<=115000]
    else:
        print(f"Filtering {name} using bureau IDs...")
        df = df[df['SK_ID_BUREAU'].isin(allowed_bureau_ids)]

    # Strings that aren't numbers will stay as strings.
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass  # Keep as original if not numeric
    
    # map Pandas types to SQLAlchemy/PostgreSQL types
    dtype_mapping = {}
    for col_name, dtype in df.dtypes.items():
        if 'float' in str(dtype):
            dtype_mapping[col_name] = types.Float
        elif 'int' in str(dtype):
            dtype_mapping[col_name] = types.Integer
            
    # Upload to Render
    df.to_sql(
        name, 
        engine, 
        if_exists="replace", 
        index=False,
        dtype=dtype_mapping
    )

    with engine.connect() as conn:
        print(f"Creating index for {name}...")
       
        index_col = 'SK_ID_CURR' if 'SK_ID_CURR' in df.columns else 'SK_ID_BUREAU'
        
        conn.execute(text(f'CREATE INDEX idx_{name}_{index_col} ON "{name}" ("{index_col}");'))
        conn.commit()

Processing table: application_test
2026-07-17 10:21:41,128 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-07-17 10:21:41,129 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-07-17 10:21:41,194 INFO sqlalchemy.engine.Engine select current_schema()
2026-07-17 10:21:41,194 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-07-17 10:21:41,253 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-07-17 10:21:41,254 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-07-17 10:21:41,307 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-17 10:21:41,311 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalo

In [10]:
# test
import psycopg
with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        # cur.execute('SELECT "SK_ID_CURR" FROM preprocessed_data LIMIT 5;')
        # print(cur.fetchall())
        # print(cur.description)

        cur.execute("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_name = 'bureau';
        """)
        print("columns: ", cur.fetchall())

        # cur.execute('SELECT * FROM credit_card_balance ORDER BY "SK_ID_CURR" LIMIT 5;')
        # print(cur.fetchall())
        # print(cur.description)
        # cur.execute("SELECT * FROM installments_payments WHERE 1=0;")
        # print("rows:", cur.fetchone())

# with engine.connect() as conn:
#     result = conn.execute(text('SELECT "SK_ID_CURR" FROM preprocessed_data WHERE "SK_ID_CURR"=100001 LIMIT 1;'))
#     # print(result)
#     # print([desc[0] for desc in result.description])
#     for row in result:
#         print("id:", row.SK_ID_CURR)

columns:  [('AMT_ANNUITY',), ('SK_ID_BUREAU',), ('AMT_CREDIT_SUM_OVERDUE',), ('DAYS_CREDIT_UPDATE',), ('SK_ID_CURR',), ('DAYS_CREDIT',), ('CREDIT_DAY_OVERDUE',), ('DAYS_CREDIT_ENDDATE',), ('DAYS_ENDDATE_FACT',), ('AMT_CREDIT_MAX_OVERDUE',), ('CNT_CREDIT_PROLONG',), ('AMT_CREDIT_SUM',), ('AMT_CREDIT_SUM_DEBT',), ('AMT_CREDIT_SUM_LIMIT',), ('CREDIT_ACTIVE',), ('CREDIT_CURRENCY',), ('CREDIT_TYPE',)]
